# Build Dimension Seller
1. read the silver sellers table
2. create the seller surrogate key
3. select the required columns
4. write the transformed data to gold dim_sellers table

In [0]:
#Imports
from pyspark.sql.functions import col,row_number
from pyspark.sql import Window

### Step1 - read the silver sellers table

In [0]:
sellers_df = spark.read.table("olist_catalog.silver.sellers")


### Step2 - create the seller surrogate key

In [0]:
window_spec=Window.orderBy("seller_id")
dim_sellers_df = sellers_df.withColumn("seller_sk",row_number().over(window_spec))

### Step3 - select the required columns

In [0]:
dim_sellers_final_df = (
    dim_sellers_df.select(
        "seller_sk",
        "seller_id",
        "seller_city",
        "seller_state"
    )
    )

### Step4 - write the transformed data to gold dim_sellers table

In [0]:
(
    dim_sellers_final_df.write
        .format("delta")
        .option("overwriteSchema","True")
        .mode("overwrite")
        .saveAsTable("olist_catalog.gold.dim_sellers")
)

In [0]:
%sql
select * from olist_catalog.gold.dim_sellers